# Demo · the whole workflow on five questions

```
18 regulations (.docx) ─► chunks ─► BM25 + dense ─► rank fusion ─► reranker ─► top 5 ─► LLM + citations
```

This notebook follows five student questions through every stage of the system, then shows the evaluation results.
It reads the saved outputs of the pipeline (`data/processed`, `data/index`, `data/runs/test`, `reports/`), so it runs
in seconds with no model or API. The last section answers **new** questions live (optional: needs the models and an
API key).

| id | question | what it tests |
|---|---|---|
| q014 | Bao nhiêu điểm thì qua môn? | the wording differs from the text (*qua môn* vs *đạt*): only meaning-based search finds it |
| q051 | Muốn chuyển ngành thì phải nộp đơn trước bao lâu? | a deadline, which the closed-book model invents |
| q066 | Làm lớp trưởng … được bao nhiêu điểm rèn luyện? | a number that sits in a table |
| q120 | Văn phòng tư vấn và hỗ trợ sinh viên ở phòng nào? | a contact detail, which the closed-book model invents |
| q123 | Chương trình tiêu chuẩn cần chuẩn đầu ra tiếng Anh IELTS mấy chấm? | **not in the documents**: the assistant must refuse |

Sections 2, 4 (detail) and 5 zoom in on the first question; the others show all five.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Markdown, display

from src.rag.dataset import load_chunks, load_gold_chunks, load_golden
from src.rag.pipeline import FUSE_DEPTH, RERANK_DEPTH, CachedRetriever, build_messages, parse_answer
from src.rag.retrieve import dense_scores, rerank, rrf, tok_syllable, top

DEMO_IDS = ["q014", "q051", "q066", "q120", "q123"]   # any test-split ids from eval/golden.jsonl
DEMO_ID = DEMO_IDS[0]                                 # the one shown in detail
RUNS = ROOT / "data" / "runs" / "test"

chunks = load_chunks()
by_id = {c["id"]: c for c in chunks}
golden = {r["id"]: r for r in load_golden()}
gold = load_gold_chunks()


def table(header, rows):
    """Render a small Markdown table."""
    out = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    out += ["| " + " | ".join(str(x).replace("|", "/") for x in r) + " |" for r in rows]
    display(Markdown("\n".join(out)))


def loc(cid):
    return f"{by_id[cid]['doc_id']} · {by_id[cid]['loc']}"


def gold_of(qid):
    return sorted({c for g in gold.get(qid, []) for c in g})

## 1 · Corpus: 18 documents from the student portal

In [ ]:
manifest = yaml.safe_load(open(ROOT / "sources.yaml", encoding="utf-8"))
n_chunks = {}
for c in chunks:
    n_chunks[c["doc_id"]] = n_chunks.get(c["doc_id"], 0) + 1
table(["id", "document", "chunks"],
      [(m["id"], m["file"].rsplit(".", 1)[0][:70], n_chunks.get(m["id"], 0)) for m in manifest])
print(f"{len(chunks)} chunks in total")

## 2 · Chunking: one chunk = one article (*Điều*) or a group of its clauses

Each chunk carries a context header (document + section path) that is embedded with the text, so it can be understood
on its own. The chunk that holds the answer to the first question:

In [ ]:
for cid in gold_of(DEMO_ID):
    c = by_id[cid]
    print(f"id: {c['id']}   location: {c['loc']}   {c['n_words']} words")
    print("-" * 100)
    print(c["context"])
    print()
    print(c["text"][:1200])

## 3 · The five questions and their reference answers (written and checked by hand)

In [ ]:
table(["id", "question", "expected", "reference answer", "evidence"],
      [(i, golden[i]["question"], golden[i]["expected_behavior"], golden[i]["answer"][:120],
        ", ".join(loc(c) for c in gold_of(i)) or "— (not in the corpus)") for i in DEMO_IDS])

## 4 · Retrieval, step by step

- **BM25** matches shared syllables, weighted by how rare they are.
- **Dense** (Vietnamese_Embedding) matches meaning.
- **Rank fusion** (RRF) merges the two ranked lists.
- **Reranker** (bge-reranker-v2-m3) reads the question and each of the top 20 together, and re-orders them. Its top 5
  are the excerpts given to the LLM.

In [ ]:
ret = CachedRetriever(chunks)


def stages(qid):
    question = golden[qid]["question"]
    lex = top(ret.ids, ret.bm25.scores(tok_syllable(question)), FUSE_DEPTH)
    den = top(ret.ids, dense_scores(ret.vecs, ret.qvec[qid]), FUSE_DEPTH)
    fused = rrf([[c for c, _ in lex], [c for c, _ in den]])
    reranked = rerank([c for c, _ in fused[:RERANK_DEPTH]], ret.scores.get(qid, {}))
    return {"BM25": lex, "dense": den, "fusion": fused, "reranked": reranked}


def rank_of(ranked, golds):
    return next((i for i, (cid, _) in enumerate(ranked, 1) if cid in golds), None)


rows = []
for i in DEMO_IDS:
    golds = set(gold_of(i))
    if not golds:
        rows.append((i, "–", "–", "–", "–", "no answer chunk exists (refusal test)"))
        continue
    st = stages(i)
    ranks = [rank_of(st[k], golds) for k in ("BM25", "dense", "fusion", "reranked")]
    rows.append((i, *(r or "> 50" for r in ranks), "✅ in the 5 excerpts" if (ranks[-1] or 99) <= 5 else "❌ missed"))
display(Markdown("**Rank of the answer chunk at each stage** (1 = first)"))
table(["id", "BM25", "dense", "fusion", "reranked", "result"], rows)

### Detail for the first question: the top 5 at each stage (✅ = the answer chunk)

In [ ]:
golds = set(gold_of(DEMO_ID))
st = stages(DEMO_ID)
for name, ranked in st.items():
    display(Markdown(f"**{name}**: answer chunk at rank **{rank_of(ranked, golds) or 'not in top 50'}**"))
    table(["rank", "", "chunk", "score"],
          [(n, "✅" if cid in golds else "", loc(cid), f"{s:.3f}") for n, (cid, s) in enumerate(ranked[:5], 1)])

## 5 · The prompt sent to the LLM (first question)

The system prompt (`prompts/rag_system.txt`) tells the model to:
- use only the numbered excerpts;
- cite them as `[n]`;
- refuse with a fixed sentence when the excerpts do not contain the answer.

In [ ]:
context = [by_id[c] for c, _ in st["reranked"][:5]]
messages = build_messages(golden[DEMO_ID], context)
print("SYSTEM:\n" + messages[0]["content"])
print("\nUSER (first 1500 characters):\n" + messages[1]["content"][:1500] + " …")

## 6 · The answers: closed book vs RAG, API model vs local model

The same questions went to four configurations. *Closed book* gets the question only. *RAG* gets the 5 retrieved
excerpts. Answerable questions are graded by the LLM judge against the reference answer. For q123 the only right
behaviour is to refuse.

In [ ]:
CONFIGS = [("closed-api", "Luna, closed book"), ("rag-api", "Luna + RAG"),
           ("closed-local", "Qwen, closed book"), ("rag-local", "Qwen + RAG")]


def saved(config, qid):
    return json.loads((RUNS / config / f"{qid}.json").read_text(encoding="utf-8"))


def outcome(config, qid):
    answer = saved(config, qid)["answer"]
    if golden[qid]["expected_behavior"] == "refuse":
        return "✅ refused" if parse_answer(answer, 5).refused else "❌ answered anyway"
    g = RUNS / "judge-api" / config / f"{qid}.json"
    grade = json.loads(g.read_text(encoding="utf-8"))["correctness"] if g.exists() else "–"
    return {"correct": "✅ correct", "partial": "◐ partial", "incorrect": "❌ incorrect",
            "refused": "❌ refused"}.get(grade, grade)


table(["id", *[label for _, label in CONFIGS]], [(i, *(outcome(c, i) for c, _ in CONFIGS)) for i in DEMO_IDS])

In [ ]:
for i in DEMO_IDS:
    display(Markdown(f"**{i} · {golden[i]['question']}**  \n*Reference:* {golden[i]['answer'][:200]}"))
    for config, label in CONFIGS[:2]:
        print(f"{label:18s}→ " + " ".join(saved(config, i)["answer"].split())[:350])
    print()

## 7 · Results on the whole test split (94 questions, run once after tuning on dev)

In [ ]:
gen = json.loads((ROOT / "reports" / "generation_test.json").read_text(encoding="utf-8"))["metrics"]
ret_res = json.loads((ROOT / "reports" / "retrieval_test.json").read_text(encoding="utf-8"))

table(["config", "overall", "multiple choice", "free text", "false refusal", "refusal recall", "grounded"],
      [(c, *(f"{100 * gen[c][k]:.0f}%" if gen[c].get(k) is not None else "–"
             for k in ("overall", "mcq_acc", "free_acc", "false_refusal", "refusal_recall", "grounded")))
       for c in ("closed-api", "rag-api", "closed-local", "rag-local")])

In [ ]:
# Chart style: recessive grid, thin bars, direct value labels, text in ink colours (not series colours).
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
BLUE, ORANGE = "#2a78d6", "#eb6834"           # categorical slots 1 and 2 of the validated palette
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
                     "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})


def grouped(ax, groups, a, b, labels, title, ylabel, fmt, ymax):
    x = np.arange(len(groups))
    w = 0.34
    for off, vals, color, lab in ((-w / 2 - 0.01, a, ORANGE, labels[0]), (w / 2 + 0.01, b, BLUE, labels[1])):
        bars = ax.bar(x + off, vals, w, color=color, label=lab, zorder=2)
        for bar, v in zip(bars, vals, strict=True):
            ax.text(bar.get_x() + bar.get_width() / 2, v + ymax * 0.015, fmt(v), ha="center", va="bottom",
                    fontsize=10, color=INK)
    ax.set_xticks(x, groups)
    ax.set_ylim(0, ymax)
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left", fontsize=12, color=INK)
    ax.yaxis.grid(True, color=GRID, zorder=0)
    ax.legend(frameon=False, loc="upper left")


def mrr(corpus, method):
    return next(r["MRR@10"] for r in ret_res if r["corpus"] == corpus and r["method"] == method)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
grouped(ax1, ["gpt-6-luna (API)", "qwen3.5:4b (local)"],
        [gen["closed-api"]["overall"], gen["closed-local"]["overall"]],
        [gen["rag-api"]["overall"], gen["rag-local"]["overall"]],
        ("closed book", "RAG"), "Overall score: retrieval more than doubles it", "score (test)",
        lambda v: f"{100 * v:.0f}%", 1.1)
ax1.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
methods = [("BM25", "BM25 (syllable)"), ("Dense", "Dense vn-embedding"), ("Hybrid", "Hybrid BM25 + vn-embedding")]
grouped(ax2, [m for m, _ in methods], [mrr("fixed-300", k) for _, k in methods],
        [mrr("structure", k) for _, k in methods], ("fixed 300-word windows", "structure-aware chunks"),
        "Retrieval MRR@10: structure-aware chunks win for every method", "MRR@10 (test)", lambda v: f"{v:.2f}", 1.1)
fig.tight_layout()
plt.show()

Full numbers, error analysis, the three tuning runs on dev and the incident log are in
[`docs/results.md`](../docs/results.md).

## 8 · Ask new questions (optional, live)

Set `RUN_LIVE = True` to answer the questions below (edit the list) with the frozen pipeline. This needs:
- torch and sentence-transformers;
- the two models in the Hugging Face cache (downloaded by `scripts/run_models.py`);
- `RAG_LLM_*` in `.env`, or Ollama running for the local model.

In [ ]:
RUN_LIVE = False
QUESTIONS = [
    "Vắng thi có phép thì nhận điểm gì và khi nào được thi lại?",
    "Bị cảnh báo học vụ mấy lần thì bị buộc thôi học?",
    "Đi thi trễ bao lâu thì không được vào phòng thi?",
    "Học bổng khuyến khích học tập cần điểm rèn luyện bao nhiêu?",
    "Thư viện trường mở cửa đến mấy giờ?",          # not in the documents: should be refused
]

if RUN_LIVE:
    from src.rag.llm import from_env
    from src.rag.pipeline import LiveRetriever, ask

    live = LiveRetriever(chunks)                    # loads the embedder and the reranker once
    llm = from_env("api")                           # or from_env("local")
    for question in QUESTIONS:
        out = ask(question, live, llm)
        print("Q:", question)
        print("A:", out["answer"])
        for n, src in out["sources"].items():
            print(f"   [{n}] {src}")
        print()